In [1]:
import sys

sys.path.append('..')

from src.graph_loader import load_graph, load_all, load_graph_types, Graph

from src.search_and_path_algorithms import bfs, printShortestDistance

In [2]:
all_graphs = load_all()

In [3]:
graph_types = load_graph_types()

In [4]:
# for graph in graph_types['Individual']:
#     print(graph.file_path)

increase-n8-2.txt - start is a red node + directed

gnm-10-20-1.txt - start is a red node + non directed

In [5]:
G = all_graphs['wall-n-10000.txt']
edges = G.edges

start_ind = G.index[G.names[G.s]]
target_ind = G.index[G.names[G.t]]

if start_ind in G.red or target_ind in G.red:
    print(-1)
    
# for edge in edges:
#     print(G.names[edge[0]], G.names[edge[1]], edge[2])

In [6]:
printShortestDistance(G)

2

In [7]:
def remove_red(g: Graph, keep_st: bool = True) -> Graph:
    """Return a copy of g without the vertices in R and their edges.

    If keep_st is True, s and t are kept even when they are red.
    Written by Claude (Anthropic)
    """
    keep = [v for v in range(g.n)
            if v not in g.red or (keep_st and v in (g.s, g.t))]
    new_id = {old: new for new, old in enumerate(keep)}   # old index -> new index

    names = [g.names[v] for v in keep]
    adj = [[] for _ in keep]
    edges = []
    for u, v, directed in g.edges:
        if u in new_id and v in new_id:                   # both ends survive
            nu, nv = new_id[u], new_id[v]
            edges.append((nu, nv, directed))
            adj[nu].append(nv)
            if not directed:
                adj[nv].append(nu)

    return Graph(
        names=names,
        index={name: i for i, name in enumerate(names)},
        red={new_id[v] for v in g.red if v in new_id},    # only red s or t can remain
        s=new_id.get(g.s, -1),                            # -1 if s was removed
        t=new_id.get(g.t, -1),
        adj=adj,
        edges=edges,
        directed=g.directed,
        file_path=g.file_path,
    )

In [8]:
G.red, G.s, G.t, G.names

({79995},
 7,
 0,
 ['0',
  '1',
  '2',
  '3',
  '4',
  '5',
  '6',
  '7',
  '8',
  '9',
  '10',
  '11',
  '12',
  '13',
  '14',
  '15',
  '16',
  '17',
  '18',
  '19',
  '20',
  '21',
  '22',
  '23',
  '24',
  '25',
  '26',
  '27',
  '28',
  '29',
  '30',
  '31',
  '32',
  '33',
  '34',
  '35',
  '36',
  '37',
  '38',
  '39',
  '40',
  '41',
  '42',
  '43',
  '44',
  '45',
  '46',
  '47',
  '48',
  '49',
  '50',
  '51',
  '52',
  '53',
  '54',
  '55',
  '56',
  '57',
  '58',
  '59',
  '60',
  '61',
  '62',
  '63',
  '64',
  '65',
  '66',
  '67',
  '68',
  '69',
  '70',
  '71',
  '72',
  '73',
  '74',
  '75',
  '76',
  '77',
  '78',
  '79',
  '80',
  '81',
  '82',
  '83',
  '84',
  '85',
  '86',
  '87',
  '88',
  '89',
  '90',
  '91',
  '92',
  '93',
  '94',
  '95',
  '96',
  '97',
  '98',
  '99',
  '100',
  '101',
  '102',
  '103',
  '104',
  '105',
  '106',
  '107',
  '108',
  '109',
  '110',
  '111',
  '112',
  '113',
  '114',
  '115',
  '116',
  '117',
  '118',
  '119',
  '120',
  '

In [9]:
G_without_red = remove_red(G)
G_without_red.edges, G_without_red.names


([(0, 1, False),
  (0, 7, False),
  (1, 2, False),
  (2, 3, False),
  (3, 8, False),
  (3, 4, False),
  (4, 5, False),
  (5, 6, False),
  (6, 7, False),
  (8, 9, False),
  (8, 15, False),
  (9, 10, False),
  (10, 11, False),
  (11, 16, False),
  (11, 12, False),
  (12, 13, False),
  (13, 14, False),
  (14, 15, False),
  (16, 17, False),
  (16, 23, False),
  (17, 18, False),
  (18, 19, False),
  (19, 24, False),
  (19, 20, False),
  (20, 21, False),
  (21, 22, False),
  (22, 23, False),
  (24, 25, False),
  (24, 31, False),
  (25, 26, False),
  (26, 27, False),
  (27, 32, False),
  (27, 28, False),
  (28, 29, False),
  (29, 30, False),
  (30, 31, False),
  (32, 33, False),
  (32, 39, False),
  (33, 34, False),
  (34, 35, False),
  (35, 40, False),
  (35, 36, False),
  (36, 37, False),
  (37, 38, False),
  (38, 39, False),
  (40, 41, False),
  (40, 47, False),
  (41, 42, False),
  (42, 43, False),
  (43, 48, False),
  (43, 44, False),
  (44, 45, False),
  (45, 46, False),
  (46, 47, Fals

In [10]:
printShortestDistance(G_without_red)

2

In [16]:
import io, time
from contextlib import redirect_stdout

# Time the pipeline (remove_red + shortest path) on every graph; output is captured so it isn't timed as printing.
timings = []
for name, g in all_graphs.items():
    buf = io.StringIO()
    start = time.perf_counter()
    with redirect_stdout(buf):
        out = printShortestDistance(remove_red(g))
    timings.append((time.perf_counter() - start, name, g.n, g.m, out))

for secs, name, n, m, out in sorted(timings, reverse=True):
    print(f"{secs*1000:10.2f} ms  n={n:<8} m={m:<9} {name}\n    {out if out != -1 else -1}")
print(f"\nTotal: {sum(t[0] for t in timings):.2f} s over {len(timings)} graphs")

    122.42 ms  n=70001    m=80000     wall-z-10000.txt
    2
     40.64 ms  n=80000    m=89999     wall-n-10000.txt
    2
     35.16 ms  n=5757     m=163671    rusty-2-5757.txt
    5
     30.73 ms  n=60002    m=70001     wall-p-10000.txt
    2
     26.31 ms  n=5000     m=123432    rusty-2-5000.txt
    5
     21.56 ms  n=4500     m=101086    rusty-2-4500.txt
    5
     16.98 ms  n=4000     m=79432     rusty-2-4000.txt
    5
     16.02 ms  n=5757     m=163671    common-2-5757.txt
    -1
     13.40 ms  n=3500     m=61442     rusty-2-3500.txt
    5
     11.89 ms  n=5000     m=123072    common-2-5000.txt
    -1
     10.09 ms  n=3000     m=45332     rusty-2-3000.txt
    5
      9.81 ms  n=4500     m=100374    common-2-4500.txt
    -1
      8.50 ms  n=4000     m=8000      gnm-4000-8000-0.txt
    6
      8.22 ms  n=4000     m=79124     common-2-4000.txt
    -1
      8.03 ms  n=5757     m=14135     bht.txt
    7
      7.91 ms  n=2000     m=4000      gnm-2000-4000-0.txt
    7
      6.99 ms  n=25

In [19]:
import csv

outputs = []
for name, g in all_graphs.items():
    temp = printShortestDistance(remove_red(g))
    outputs.append((name, temp))

for name, out in outputs:
    print(f"{name} => {out}")

with open("../results.txt", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "None"])
    writer.writerows(outputs)

G-ex.txt => 4
P3.txt => -1
bht.txt => 7
common-1-100.txt => -1
common-1-1000.txt => -1
common-1-1500.txt => -1
common-1-20.txt => -1
common-1-2000.txt => -1
common-1-250.txt => -1
common-1-2500.txt => -1
common-1-3000.txt => -1
common-1-3500.txt => -1
common-1-4000.txt => -1
common-1-4500.txt => -1
common-1-50.txt => -1
common-1-500.txt => -1
common-1-5000.txt => -1
common-1-5757.txt => -1
common-2-100.txt => -1
common-2-1000.txt => -1
common-2-1500.txt => -1
common-2-20.txt => -1
common-2-2000.txt => -1
common-2-250.txt => -1
common-2-2500.txt => -1
common-2-3000.txt => -1
common-2-3500.txt => -1
common-2-4000.txt => -1
common-2-4500.txt => -1
common-2-50.txt => -1
common-2-500.txt => -1
common-2-5000.txt => -1
common-2-5757.txt => -1
dodecahedron.txt => -1
gnm-10-15-0.txt => 2
gnm-10-15-1.txt => -1
gnm-10-20-0.txt => -1
gnm-10-20-1.txt => -1
gnm-1000-1500-0.txt => -1
gnm-1000-1500-1.txt => -1
gnm-1000-2000-0.txt => 8
gnm-1000-2000-1.txt => -1
gnm-2000-3000-0.txt => 9
gnm-2000-3000-1.